<a href="https://colab.research.google.com/github/Ewlisten/Basketball-Motion/blob/main/BBALL_Motion_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

-----------------------------------
INSTALL
------------------------------

In [ ]:
!pip install -q roboflow ultralytics opencv-python scikit-learn supervision torchreid

import torch
import math
import numpy as np
from roboflow import Roboflow
from ultralytics import YOLO
import cv2
import supervision as sv
from collections import defaultdict
from pathlib import Path
from sklearn.cluster import KMeans


print(f"GPU available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

import requests
import base64

class SimpleInferenceClient:
    """Drop-in replacement for inference_sdk.InferenceHTTPClient — pure HTTP, no version constraints."""
    def __init__(self, api_key, api_url="https://serverless.roboflow.com"):
        self.api_key = api_key
        self.api_url = api_url

    def infer(self, image_bytes, model_id):
        b64 = base64.b64encode(image_bytes).decode("utf-8")
        url = f"{self.api_url}/{model_id}"
        resp = requests.post(
            url,
            params={"api_key": self.api_key},
            data=b64,
            headers={"Content-Type": "application/x-www-form-urlencoded"},
        )
        resp.raise_for_status()
        return resp.json()

court_client = SimpleInferenceClient(api_key="2zWr1IWK0uNCszPDo3Ob")

COURT_MODEL_ID       = "basketball-court-detection-2/22"
COURT_API_KEY         = "2zWr1IWK0uNCszPDo3Ob"
COURT_CONF_THRESH     = 0.5
COURT_UPDATE_EVERY    = 8      # re-run keypoint detection every N frames
COURT_MIN_POINTS      = 6      # stricter: need ≥6 matched points for a trustworthy homography
ENABLE_COURT_FEATURES = True   # flip off to skip court API calls entirely (minimap/analytics only)

# ── Court keypoint persistence / EMA (minimap stability) ──
COURT_EMA_ALPHA       = 0.4    # EMA weight for new keypoints (lower = more smoothing)
COURT_MAX_AGE_FRAMES  = 60     # drop a persisted keypoint if unseen for this many frames

court_client = SimpleInferenceClient(api_key=COURT_API_KEY)

COURT_KEYPOINT_TARGETS = {
    0:  (0, 50),  7:  (0, 47), 15: (47, 50), 16: (47, 25),
    19: (65, 25), 22: (75, 25), 26: (88.75, 25),
    21: (75, 33), 23: (75, 17), 24: (72, 42), 25: (72, 8),
    12: (35, 50), 18: (94, 50), 27: (94, 50), 28: (94, 47),
    29: (94, 33), 30: (94, 25), 31: (94, 0),
}


MODELS DOWNLOADS
-------------------

In [ ]:


rf = Roboflow(api_key="2zWr1IWK0uNCszPDo3Ob")
project = rf.workspace("basketball-motion").project("basketball-motion-owdql")
version4 = project.version(4) #V4 = 4000 imgs (stable ball/rim)
dataset4 = version4.download("yolov8")

# Load a model
model_v4 = YOLO('yolov8n.pt')  # load a pretrained model (recommended for training)
model_v4.train(
    data='/content/Basketball-Motion,-4/data.yaml',
        epochs=30,
            imgsz=960,                 #COULD BE 1280 FOR DETAIL
                #fraction=.75,
                    patience=15,
                        device=DEVICE,      # GPU acceleration
                            batch=16,           # larger batch on GPU (was 16), CAN BE 8 FOR DETAIL, COST TIME
                                workers=4,          # parallel data loading
                                    amp=True,           # mixed precision (FP16) — faster on T4
                                        cache=False,        # cache images in RAM for faster epochs - changed to False to reduce RAM usage
                                        )


rf = Roboflow(api_key="2zWr1IWK0uNCszPDo3Ob")
project = rf.workspace("basketball-motion").project("basketball-motion-owdql")
version5 = project.version(5) # V5 = 640 imgs (sharpest),
dataset5 = version5.download("yolov8")

# Load a model
model_v5 = YOLO('yolov8n.pt')  # load a pretrained model (recommended for training)
model_v5.train(
    data='/content/Basketball-Motion,-5/data.yaml',
        epochs=40,
            imgsz=640,
                patience=10,
                    device=DEVICE,      # GPU acceleration
                        batch=32,           # larger batch on GPU (was 16)
                            workers=4,          # parallel data loading
                                amp=True,           # mixed precision (FP16) — faster on T4
                                    cache=False,        # cache images in RAM for faster epochs - changed to False to reduce RAM usage
                                    )

IMPORTS & PARAMETERS
-------------------

In [ ]:


"""IMPORTS & PARAMETERS
--------------------------------------
"""

# ---- Paths ----
MODEL_PATH   = "MODEL PATH"
VIDEO_PATH   = "VIDEO PATH"
OUTPUT_VIDEO = "OUTPUT PATH"

# ---- Detection ----
CONF_THRESH      = 0.05   # lower = more detections, more false positives
IMGSZ            = 640   # inference resolution (higher = better on small ball)

# ---- Ball tracking ----
TRAIL_LENGTH     = 35     # frames of fading dot trail behind ball
MIN_TRACK_LEN    = 1     # minimum detections to attempt a parabola fit
MAX_BALL_JUMP_PX = 320     # max px distance ball can move between frames without reset
BALL_MIN_Y_FRACTION = 0.10   # ignore balls detected in top 10% of frame (crowd/scoreboard)
BALL_MAX_Y_FRACTION = 0.90   # ignore balls below 90% of frame (floor artifacts)

# ---- Ball-in-play gating (local proxy — no API dependency) ----
BALL_GATE_PAD_PX    = 80    # padding around player/rim union box for ball validity
BALL_GATE_MIN_DETS  = 2     # need at least this many player detections to build the gate

# ---- Shot detection ----
POST_APEX_FRAMES = 3      # how many frames after apex to accept rim overlap as a make
RIM_EXPAND_PX    = 5     # expand rim bbox by this many pixels for overlap check (forgiveness)
MIN_R_SQUARED    = 0.85   # min parabola fit to trust shot detection
RIM_SMOOTH_ALPHA = 0.35   #EMA weight for rim detection (lower = more smoothing)
MIN_ARC_HEIGHT_PX = 40    # Min vertical extent before ball is treated as a shot
# ---- Overlay style ----
OVERLAY_DURATION = 90     # frames to show MADE / MISSED banner
ARC_COLOR_MAKE   = (0,   220,  80)   # green
ARC_COLOR_MISS   = (0,    60, 220)   # red
ARC_COLOR_LIVE   = (255, 255, 255)   # white (arc color before outcome is known)
TRAIL_COLOR      = (255, 200,   0)   # amber ball trail
RIM_COLOR        = (0,   165, 255)   # orange rim box


In [ ]:
model_v5 = YOLO('/content/runs/detect/train-2/weights/best.pt')
model_v4 = YOLO('/content/runs/detect/train/weights/best.pt')

# What each model contributes
V5_CLASSES = ['player']           # trust V5 for players only
V4_CLASSES = ['ball', 'rim']      # trust V4 for ball and rim only

def predict_merged(source, conf_player=0.40, conf_ball=0.15):
    """
    Run both models and merge detections by class responsibility.
    Returns a list (one entry per frame) of detection dicts.
    """
    results_v5 = model_v5.predict(source=source, imgsz=640,  conf=conf_player, device=DEVICE, stream=True, half=True, verbose=False)
    results_v4 = model_v4.predict(source=source, imgsz=1280, conf=conf_ball,   device=DEVICE, stream=True, half=True, verbose=False)

    merged_frames = []

    for r5, r4 in zip(results_v5, results_v4):
        frame_dets = []

        # Pull players from V5
        for box in (r5.boxes or []):
            name = model_v5.names[int(box.cls[0])]
            if name in V5_CLASSES:
                x1,y1,x2,y2 = map(int, box.xyxy[0].tolist())
                frame_dets.append({
                    "class":  name,
                    "conf":   float(box.conf[0]),
                    "xyxy":   [x1,y1,x2,y2],
                    "center": [int((x1+x2)/2), int((y1+y2)/2)],
                    "source": "v5"
                })

        # Pull ball + rim from V4
        for box in (r4.boxes or []):
            name = model_v4.names[int(box.cls[0])]
            if name in V4_CLASSES:
                x1,y1,x2,y2 = map(int, box.xyxy[0].tolist())
                frame_dets.append({
                    "class":  name,
                    "conf":   float(box.conf[0]),
                    "xyxy":   [x1,y1,x2,y2],
                    "center": [int((x1+x2)/2), int((y1+y2)/2)],
                    "source": "v4"
                })

        merged_frames.append(frame_dets)

    return merged_frames

HELPERS
---------------

In [ ]:

def _three_point_arc(basket_x, num_pts=30):
    radius = 23.75
    corner_y_near, corner_y_far = 3, 47
    baseline_x = 0 if basket_x < 47 else 94
    dy = 25 - corner_y_near
    dx = math.sqrt(radius**2 - dy**2)
    x_transition = basket_x + dx if basket_x < 47 else basket_x - dx
    theta1 = math.atan2(corner_y_near - 25, x_transition - basket_x)
    theta2 = math.atan2(corner_y_far - 25, x_transition - basket_x)
    if basket_x < 47:
        thetas = np.linspace(theta1, theta1 + (2*math.pi - (theta1 - theta2) % (2*math.pi)), num_pts)
    else:
        thetas = np.linspace(theta2, theta2 + (2*math.pi - (theta2 - theta1) % (2*math.pi)), num_pts)
    arc_pts = [(basket_x + radius*math.cos(t), 25 + radius*math.sin(t)) for t in thetas]
    return [(baseline_x, corner_y_near), (x_transition, corner_y_near)] + arc_pts + \
           [(x_transition, corner_y_far), (baseline_x, corner_y_far)]

def _restricted_area(basket_x, num_pts=15):
    radius = 4
    start_angle = math.pi/2 if basket_x < 47 else -math.pi/2
    end_angle   = -math.pi/2 if basket_x < 47 else math.pi/2
    thetas = np.linspace(start_angle, end_angle, num_pts)
    return [(basket_x + radius*math.cos(t), 25 + radius*math.sin(t)) for t in thetas]

def _circle(cx, cy, radius, num_pts=40):
    return [(cx + radius*math.cos(t), cy + radius*math.sin(t)) for t in np.linspace(0, 2*math.pi, num_pts)]

def court_lines():
    left_basket_x, right_basket_x = 5.25, 88.75
    return [
        [(0,0),(94,0),(94,50),(0,50),(0,0)],
        [(47,0),(47,50)],
        [(0,17),(19,17),(19,33),(0,33)],
        [(94,17),(75,17),(75,33),(94,33)],
        _circle(47, 25, 6),
        _circle(19, 25, 6),
        _circle(75, 25, 6),
        _three_point_arc(left_basket_x),
        _three_point_arc(right_basket_x),
        _restricted_area(left_basket_x),
        _restricted_area(right_basket_x),
    ]

In [ ]:

def get_video_info(path):
    """Return (fps, width, height, total_frames) for a video file."""
    cap = cv2.VideoCapture(path)
    fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0
    w      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return fps, w, h, total

def fit_parabola(frame_indices, ys, fps):
    """
    Fit a quadratic y(t) = a*t^2 + b*t + c to the ball's vertical pixel positions.
    Returns (poly_coefficients, poly1d_object, r_squared).
    """
    frames = np.array(frame_indices, dtype=float)
    times  = (frames - frames[0]) / fps
    ys_arr = np.array(ys, dtype=float)
    poly   = np.polyfit(times, ys_arr, 2)
    p       = np.poly1d(poly)
    ss_res  = np.sum((ys_arr - p(times)) ** 2)
    f = ys_arr
    ss_tot  = np.sum((ys_arr - np.mean(ys_arr)) ** 2)
    r_sq    = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0
    return poly, p, times, r_sq

def predict_arc_points(frame_indices, xs, ys, fps, extend_frames=0):
    poly, p, times, _r_sq = fit_parabola(frame_indices, ys, fps)
    frames = np.array(frame_indices, dtype=float)
    t_end  = (frames[-1] - frames[0]) / fps + extend_frames / fps
    ts     = np.linspace(0, t_end, num=80)
    pts    = []
    for t in ts:
        py = int(p(t))
        # linear x interpolation; clamp beyond last observed frame
        t_clamped = min(t, times[-1])
        px = int(np.interp(t_clamped, times, np.array(xs, dtype=float)))
        pts.append((px, py))
    return pts, poly

import math
import numpy as np

def is_made_shot(track, poly, fps, rim_bbox, times, xs,
                 # ---- tunable thresholds ----
                 x_tol_ratio=0.35,       # ball center must be within 35% of rim width from center
                 y_entry_above=0.3,      # how far above rim top to start the entry zone (× rim_h)
                 y_entry_below=0.8,      # how far below rim bottom for exit zone (× rim_h)
                 bounce_check_frames=8,  # how many frames after rim event to check
                 bounce_up_thresh=2,     # how many rising frames = bounce
                 extrap_max_sec=0.0,     # max extrapolation beyond last detection
                 ):
    """
    Determines if a ball track represents a made shot.
    Uses rim-crossing detection with post-rim trajectory verification.
    """
    a, b, c = poly
    x1, y1, x2, y2 = rim_bbox

    rim_cx = (x1 + x2) / 2
    rim_cy = (y1 + y2) / 2
    rim_w  = x2 - x1
    rim_h  = y2 - y1

    frames = np.array([t[0] for t in track])

    # ------------------------------------------------------------------ #
    # Find apex time                                                      #
    # ------------------------------------------------------------------ #
    if abs(a) < 1e-6:
        return False, "no_arc", None

    t_apex = -b / (2 * a)

    # ------------------------------------------------------------------ #
    # METHOD 1 — Direct detection near rim (tightened + post-rim check)   #
    # ------------------------------------------------------------------ #
    x_tolerance = rim_w * x_tol_ratio

    after_idx = np.where(times >= t_apex)[0]

    for idx in after_idx:
        cx = track[idx][1]
        cy = track[idx][2]
        t  = times[idx]

        # Tight horizontal: ball center within ~35% of rim width from center
        in_x = abs(cx - rim_cx) <= x_tolerance

        # Vertical: ball is crossing through the rim plane
        # (between slightly above rim top and slightly below rim bottom)
        y_top = y1 - rim_h * y_entry_above
        y_bot = y2 + rim_h * y_entry_below
        in_y = (cy >= y_top) and (cy <= y_bot)

        if not (in_x and in_y):
            continue

        # Ball must be descending
        vy = 2 * a * t + b
        if vy <= 0:
            continue

        # ---- POST-RIM TRAJECTORY CHECK ----
        # Look at the next N frames. A made shot: ball keeps falling
        # or disappears (net). A miss: ball bounces up or flies sideways.
        post_start = idx + 1
        post_end = min(idx + 1 + bounce_check_frames, len(track))
        post_track = track[post_start:post_end]

        if len(post_track) >= 3:
            upward_count = 0
            lateral_drift = 0

            for j in range(1, len(post_track)):
                prev_cy = post_track[j - 1][2]
                curr_cy = post_track[j][2]
                prev_cx = post_track[j - 1][1]
                curr_cx = post_track[j][1]

                # Ball moving upward = bounce off rim
                if curr_cy < prev_cy - 1:  # small threshold to ignore jitter
                    upward_count += 1

                lateral_drift += abs(curr_cx - prev_cx)

            # If ball reverses upward, it bounced — miss
            if upward_count >= bounce_up_thresh:
                continue  # don't return yet, might find a real crossing later

            # If ball drifts far laterally after "crossing," it deflected — miss
            avg_lateral = lateral_drift / len(post_track)
            if avg_lateral > rim_w * 0.5:
                continue

        elif len(post_track) == 0:

            return True, "detected_vanish", int(frames[idx])

        # Passed all checks
        return True, "detected_overlap", int(frames[idx])

    # ------------------------------------------------------------------ #
    # METHOD 2 — Parabola extrapolation (dunks, layups, occluded shots)   #
    # Same tighter tolerances                                              #
    # ------------------------------------------------------------------ #
    discriminant = b**2 - 4 * a * (c - rim_cy)

    if discriminant >= 0:
        sqrt_disc = math.sqrt(discriminant)
        t_roots = [
            (-b + sqrt_disc) / (2 * a),
            (-b - sqrt_disc) / (2 * a),
        ]

        for t_rim in t_roots:
            t_max = times[-1]
            if t_rim < t_apex:
                continue
            if t_rim > t_max + extrap_max_sec:
                continue

            # Interpolate/extrapolate x at rim crossing time
            if t_rim <= t_max:
                x_at_rim = float(np.interp(t_rim, times, xs))
            else:
                if len(xs) >= 2:
                    dt = times[-1] - times[-2]
                    if dt > 0:
                        vx = (xs[-1] - xs[-2]) / dt
                        x_at_rim = xs[-1] + vx * (t_rim - times[-1])
                    else:
                        x_at_rim = xs[-1]
                else:
                    x_at_rim = xs[-1]

            # TIGHTER x check
            if abs(x_at_rim - rim_cx) > x_tolerance:
                continue

            vy_at_rim = 2 * a * t_rim + b
            if vy_at_rim <= 0:
                continue

            # For extrapolated shots, apply an additional check:
            # if the ball was still being tracked AFTER the predicted
            # rim crossing, verify it didn't bounce
            rim_frame_approx = frames[0] + t_rim * fps
            post_rim_pts = [(t[0], t[1], t[2]) for t in track
                           if t[0] > rim_frame_approx]

            if len(post_rim_pts) >= 3:
                upward = 0
                for j in range(1, len(post_rim_pts)):
                    if post_rim_pts[j][2] < post_rim_pts[j-1][2] - 1:
                        upward += 1
                if upward >= bounce_up_thresh:
                    continue  # bounced — miss

            approx_frame = int(rim_frame_approx)
            return True, "extrapolated", approx_frame

    return False, "miss", None

In [ ]:
def draw_fading_trail(frame, trail_points, color=TRAIL_COLOR, max_radius=7):
    """Draw a fading dot trail. trail_points is a deque of (cx, cy) oldest→newest."""
    n = len(trail_points)
    for i, (cx, cy) in enumerate(trail_points):
        alpha  = (i + 1) / n
        radius = max(2, int(max_radius * alpha))
        bright = tuple(int(c * alpha) for c in color)
        cv2.circle(frame, (cx, cy), radius, bright, -1)

def draw_arc(frame, arc_pts, color, thickness=2):
    """Draw a smooth polyline arc from a list of (x, y) points."""
    for i in range(len(arc_pts) - 1):
        p1, p2 = arc_pts[i], arc_pts[i + 1]
        # skip points outside frame bounds
        h, w = frame.shape[:2]
        if not (0 <= p1[0] < w and 0 <= p1[1] < h):
            continue
        if not (0 <= p2[0] < w and 0 <= p2[1] < h):
            continue
        cv2.line(frame, p1, p2, color, thickness, lineType=cv2.LINE_AA)

def draw_make_miss_banner(frame, text, color, alpha=1.0):
    """Render a large centered MADE / MISSED banner with a semi-transparent background."""
    h, w = frame.shape[:2]
    font       = cv2.FONT_HERSHEY_DUPLEX
    font_scale = 2.8
    thickness  = 5
    (tw, th), baseline = cv2.getTextSize(text, font, font_scale, thickness)
    tx = (w - tw) // 2
    ty = int(h * 0.18)
    # dark pill background
    pad = 20
    overlay = frame.copy()
    cv2.rectangle(overlay,
                  (tx - pad, ty - th - pad),
                  (tx + tw + pad, ty + baseline + pad),
                  (20, 20, 20), -1)
    cv2.addWeighted(overlay, 0.55 * alpha, frame, 1 - 0.55 * alpha, 0, frame)
    # text
    cv2.putText(frame, text, (tx, ty), font, font_scale,
                tuple(int(c * alpha) for c in color), thickness, cv2.LINE_AA)



In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 12 — Court keypoint detection + homography + ball-in-play gate + radar minimap
# ══════════════════════════════════════════════════════════════════════

def get_court_keypoints(frame, conf_thresh=COURT_CONF_THRESH):
    """Query the court model. Returns [(class_id, x, y, conf), ...]. Fails soft on network errors."""
    try:
        _, buf = cv2.imencode(".jpg", frame)
        result = court_client.infer(buf.tobytes(), model_id=COURT_MODEL_ID)
        preds = result.get("predictions")
        if not preds:
            return []
        return [
            (kp["class_id"], kp["x"], kp["y"], kp["confidence"])
            for kp in preds[0].get("keypoints", [])
            if kp["confidence"] >= conf_thresh
        ]
    except Exception as e:
        print(f"  [court] inference failed this frame: {e}")
        return []


class ViewTransformer:
    def __init__(self, source, target):
        self.m, _ = cv2.findHomography(source.astype(np.float32), target.astype(np.float32), cv2.RANSAC, 5.0)

    def transform_points(self, points):
        if points.size == 0 or self.m is None:
            return points
        pts = points.reshape(-1, 1, 2).astype(np.float32)
        return cv2.perspectiveTransform(pts, self.m).reshape(-1, 2)


class CourtKeypointStore:
    """
    Persists court keypoints across frames with EMA smoothing.
    Keypoints that haven't been seen recently are aged out.
    Used ONLY for minimap / analytics — never for ball gating.
    """
    def __init__(self, alpha=COURT_EMA_ALPHA, max_age=COURT_MAX_AGE_FRAMES):
        self.alpha    = alpha
        self.max_age  = max_age
        self.points   = {}   # class_id -> (x, y, conf)
        self.last_seen = {}  # class_id -> frame_idx

    def update(self, raw_keypoints, frame_idx):
        """Merge new detections into the persistent store."""
        for cid, x, y, conf in raw_keypoints:
            if cid in self.points:
                ox, oy, oc = self.points[cid]
                nx = self.alpha * x + (1 - self.alpha) * ox
                ny = self.alpha * y + (1 - self.alpha) * oy
                nc = self.alpha * conf + (1 - self.alpha) * oc
                self.points[cid] = (nx, ny, nc)
            else:
                self.points[cid] = (x, y, conf)
            self.last_seen[cid] = frame_idx

        # Age out stale keypoints
        stale = [cid for cid, last in self.last_seen.items()
                 if frame_idx - last > self.max_age]
        for cid in stale:
            del self.points[cid]
            del self.last_seen[cid]

    def get_all(self):
        """Return list of (class_id, x, y, conf) for all live keypoints."""
        return [(cid, x, y, c) for cid, (x, y, c) in self.points.items()]


def build_court_transformer(court_points, targets=COURT_KEYPOINT_TARGETS, min_points=COURT_MIN_POINTS):
    """
    Returns transformer_or_None.
    Only builds a homography when enough keypoints match known court targets.
    No longer returns a pixel_polygon — ball gating is handled locally.
    """
    matched_src, matched_tgt = [], []
    for cid, x, y, conf in court_points:
        if cid in targets:
            matched_src.append([x, y])
            matched_tgt.append(targets[cid])

    if len(matched_src) < min_points:
        return None

    return ViewTransformer(np.array(matched_src), np.array(matched_tgt))


# ── Ball-in-play gate (local proxy — zero API calls) ──────────────────

def ball_in_play_region(ball_cx, ball_cy, player_boxes, rim_bbox,
                        pad_px=BALL_GATE_PAD_PX,
                        min_dets=BALL_GATE_MIN_DETS,
                        frame_h=None,
                        y_min_frac=None, y_max_frac=None):

    # ── 1. Frame-band filter ──
    if frame_h is not None and y_min_frac is not None and y_max_frac is not None:
        y_min = frame_h * y_min_frac
        y_max = frame_h * y_max_frac
        if ball_cy < y_min or ball_cy > y_max:
            return False

    # ── 2. Player/rim union box ──
    if len(player_boxes) < min_dets:
        return True   # too few references — don't filter, avoid false negatives

    # Gather all reference points (player boxes + rim if available)
    all_x1, all_y1, all_x2, all_y2 = [], [], [], []
    for (px1, py1, px2, py2) in player_boxes:
        all_x1.append(px1)
        all_y1.append(py1)
        all_x2.append(px2)
        all_y2.append(py2)

    if rim_bbox is not None:
        rx1, ry1, rx2, ry2 = rim_bbox
        all_x1.append(rx1)
        all_y1.append(ry1)
        all_x2.append(rx2)
        all_y2.append(ry2)

    # Bounding rectangle of all references, expanded by padding
    union_x1 = min(all_x1) - pad_px
    union_y1 = min(all_y1) - pad_px
    union_x2 = max(all_x2) + pad_px
    union_y2 = max(all_y2) + pad_px

    return union_x1 <= ball_cx <= union_x2 and union_y1 <= ball_cy <= union_y2


# ---- radar minimap ----
RADAR_W, RADAR_H = 300, 160
RADAR_SCALE = RADAR_W / 94
RADAR_MARGIN = 10

def _court_to_radar_px(pt):
    x, y = pt
    return int(x * RADAR_SCALE), int(RADAR_H - y * RADAR_SCALE)

def build_radar_background():
    radar = np.full((RADAR_H, RADAR_W, 3), (40, 90, 40), dtype=np.uint8)
    for line in court_lines():
        pix = [_court_to_radar_px(p) for p in line]
        for i in range(len(pix) - 1):
            cv2.line(radar, pix[i], pix[i + 1], (210, 210, 210), 1, cv2.LINE_AA)
    return radar

RADAR_BG = build_radar_background()

def draw_radar_frame(radar_points, ball_court_xy, team_colors):
    radar = RADAR_BG.copy()
    for p in radar_points:
        px = _court_to_radar_px(p["court_xy"])
        color = team_colors.get(p["team"], (160, 160, 160))
        cv2.circle(radar, px, 4, color, -1)
    if ball_court_xy is not None:
        cv2.circle(radar, _court_to_radar_px(ball_court_xy), 3, (0, 200, 255), -1)
    return radar

In [ ]:

import json
import math
import numpy as np
import cv2
import torch
import torch.nn.functional as F
from pathlib import Path
from collections import Counter, defaultdict, deque
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture


# ══════════════════════════════════════════════════════════
#  Brightness extraction
# ══════════════════════════════════════════════════════════

def extract_jersey_brightness(frame, xyxy, torso_top=0.15, torso_bottom=0.55, center_frac=0.6):
    """
    Median V (brightness) of the torso band after filtering out skin-toned
    and court-floor pixels. Same metric as the original (median), just
    removing non-jersey contamination first.
    """
    x1, y1, x2, y2 = [int(v) for v in xyxy]
    h, w = y2 - y1, x2 - x1
    if h <= 0 or w <= 0:
        return None

    top    = max(y1 + int(h * torso_top), 0)
    bottom = min(y1 + int(h * torso_bottom), frame.shape[0])
    cx     = x1 + w // 2
    half_w = int(w * center_frac / 2)
    left   = max(cx - half_w, 0)
    right  = min(cx + half_w, frame.shape[1])

    if bottom <= top or right <= left:
        return None

    crop = frame[top:bottom, left:right]
    if crop.size == 0:
        return None

    hsv  = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV).astype(np.float64)
    h_ch = hsv[..., 0]   # hue: 0-180 in OpenCV
    s_ch = hsv[..., 1]   # saturation: 0-255
    v_ch = hsv[..., 2]   # value/brightness: 0-255

    # ── Step 1: Exclude extreme pixels (near-black, near-white clipping) ──
    valid = (v_ch > 15) & (v_ch < 245)


    skin_or_floor = ((h_ch < 25) | (h_ch > 165)) & (s_ch > 30) & (s_ch < 170) & (v_ch > 50)

    jersey_mask = valid & (~skin_or_floor)

    # Need enough jersey pixels to be reliable
    if jersey_mask.sum() < max(10, 0.15 * valid.sum()):
        # Filter was too aggressive — fall back to all valid pixels
        jersey_mask = valid

    if jersey_mask.sum() < 10:
        return None

    # ── MEDIAN (same as original) — NOT 75th percentile ──
    return float(np.median(v_ch[jersey_mask]))

# ══════════════════════════════════════════════════════════
#  Crop-region diagnostic — run once, pick the best region by
#  gap/spread, not by eyeballing the raw gap
# ══════════════════════════════════════════════════════════

def diagnose_crop_regions(video_path, all_merged_detections, n_sample_frames=120, total_frames=None):
    regions = [
        ("torso_only",  0.15, 0.55, 0.6),
        ("torso_wide",  0.15, 0.55, 0.8),
        ("full_upper",  0.10, 0.75, 0.6),
        ("chest_tight", 0.20, 0.40, 0.5),
    ]
    step  = max(1, min(4, n_sample_frames // 30))
    upper = min(n_sample_frames, total_frames or 10**9, len(all_merged_detections))
    cap_s = cv2.VideoCapture(video_path)

    frames_cache = []
    for fi in range(0, upper, step):
        cap_s.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ret, sf = cap_s.read()
        if ret:
            frames_cache.append((fi, sf))
    cap_s.release()

    best_name, best_score = None, -1.0
    print("  Crop region diagnostic:")
    for name, top, bot, cf in regions:
        vs = []
        for fi, sf in frames_cache:
            for det in all_merged_detections[fi]:
                if det["class"] == "player":
                    v = extract_jersey_brightness(sf, det["xyxy"], top, bot, cf)
                    if v is not None:
                        vs.append(v)
        if len(vs) < 10:
            print(f"    {name:12s}  insufficient samples ({len(vs)})")
            continue
        vs_arr = np.array(vs).reshape(-1, 1)
        km = KMeans(n_clusters=2, n_init=5, random_state=42).fit(vs_arr)
        centers = sorted(km.cluster_centers_.flatten())
        labels = km.labels_
        gap = centers[1] - centers[0]
        spread = np.mean([vs_arr[labels == 0].std(), vs_arr[labels == 1].std()]) or 1.0
        score = gap / spread
        print(f"    {name:12s}  n={len(vs):4d}  centers=[{centers[0]:.1f}, {centers[1]:.1f}]  "
              f"gap={gap:.1f}  spread={spread:.1f}  gap/spread={score:.2f}")
        if score > best_score:
            best_score, best_name = score, (top, bot, cf, name)

    if best_name is not None:
        top, bot, cf, name = best_name
        print(f"  -> Best region: {name} (gap/spread={best_score:.2f})")
        return {"torso_top": top, "torso_bottom": bot, "center_frac": cf}
    print("  -> Diagnostic inconclusive, using defaults.")
    return {"torso_top": 0.15, "torso_bottom": 0.55, "center_frac": 0.6}


# ══════════════════════════════════════════════════════════
#  GMM threshold fit — accounts for unequal cluster spread,
#  unlike a plain KMeans midpoint
# ══════════════════════════════════════════════════════════

def fit_brightness_threshold(video_path, all_merged_detections, crop_kwargs,
                             n_sample_frames=120, total_frames=None, use_gmm=True):
    vs    = []
    step  = max(1, min(4, n_sample_frames // 30))
    upper = min(n_sample_frames, total_frames or 10**9, len(all_merged_detections))
    cap_s = cv2.VideoCapture(video_path)

    for fi in range(0, upper, step):
        cap_s.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ret, sf = cap_s.read()
        if not ret:
            continue
        for det in all_merged_detections[fi]:
            if det["class"] == "player":
                v = extract_jersey_brightness(sf, det["xyxy"], **crop_kwargs)
                if v is not None:
                    vs.append(v)
    cap_s.release()

    if len(vs) < 10:
        print(f"  Not enough brightness samples ({len(vs)}).")
        return None

    vs = np.array(vs).reshape(-1, 1)

    if use_gmm:
        gmm = GaussianMixture(n_components=2, n_init=10, random_state=42).fit(vs)
        means = gmm.means_.flatten()
        stds  = np.sqrt(gmm.covariances_.flatten())
        order = np.argsort(means)
        m0, m1 = means[order]
        s0, s1 = stds[order]
        s0, s1 = max(s0, 1e-3), max(s1, 1e-3)

        xs = np.linspace(m0, m1, 500)
        p0 = np.exp(-0.5 * ((xs - m0) / s0) ** 2) / s0
        p1 = np.exp(-0.5 * ((xs - m1) / s1) ** 2) / s1
        threshold = float(xs[np.argmin(np.abs(p0 - p1))])

        print(f"  GMM fit on {len(vs)} samples: "
              f"dark(μ={m0:.1f},σ={s0:.1f}) light(μ={m1:.1f},σ={s1:.1f}) "
              f"threshold={threshold:.1f}")
    else:
        km = KMeans(n_clusters=2, n_init=10, random_state=42).fit(vs)
        centers = sorted(km.cluster_centers_.flatten())
        threshold = (centers[0] + centers[1]) / 2
        print(f"  KMeans fit: dark≈{centers[0]:.1f}, light≈{centers[1]:.1f}, "
              f"threshold={threshold:.1f}")

    return threshold


def classify_by_brightness_weighted(v, threshold, dark_team_id=0, light_team_id=1, dead_zone=8.0):
    """
    Returns (team_id, confidence). confidence in [0,1], saturating at 4x dead_zone.
    Votes inside the dead_zone are ambiguous (-1, 0.0).
    """
    if v is None or threshold is None:
        return -1, 0.0
    d = v - threshold
    if abs(d) < dead_zone:
        return -1, 0.0
    team_id = light_team_id if d > 0 else dark_team_id
    confidence = min(1.0, abs(d) / (dead_zone * 4))
    return team_id, confidence


# ══════════════════════════════════════════════════════════
#  Person Re-ID embedder (secondary signal, layered on top)
# ══════════════════════════════════════════════════════════

class ReIDEmbedder:
    def __init__(self, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        try:
            from torchreid.utils import FeatureExtractor
            self.extractor = FeatureExtractor(
                model_name="osnet_x0_25", model_path="", device=self.device,
            )
            self.mode = "torchreid"
            print(f"  ReID: OSNet on {self.device}")
        except ImportError:
            import torchvision
            m = torchvision.models.resnet18(weights="IMAGENET1K_V1")
            m.fc = torch.nn.Identity()
            self.model = m.eval().to(self.device)
            self.mode = "resnet"
            print(f"  ReID: torchreid unavailable — ResNet18 fallback on {self.device}")

    def embed(self, crops):
        if not crops:
            return None
        if self.mode == "torchreid":
            feats = self.extractor(crops)
        else:
            batch = []
            for c in crops:
                c = cv2.resize(c, (128, 256))
                c = cv2.cvtColor(c, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
                batch.append(torch.from_numpy(c).permute(2, 0, 1))
            x = torch.stack(batch).to(self.device)
            mean = torch.tensor([0.485, 0.456, 0.406], device=self.device).view(1, 3, 1, 1)
            std  = torch.tensor([0.229, 0.224, 0.225], device=self.device).view(1, 3, 1, 1)
            with torch.no_grad():
                feats = self.model((x - mean) / std)
        return F.normalize(feats.float(), dim=1).cpu()


def crop_player(frame, xyxy):
    x1, y1, x2, y2 = [int(v) for v in xyxy]
    x1, y1 = max(x1, 0), max(y1, 0)
    x2, y2 = min(x2, frame.shape[1]), min(y2, frame.shape[0])
    if x2 <= x1 or y2 <= y1:
        return None
    crop = frame[y1:y2, x1:x2]
    return crop if crop.size else None


def fit_team_embeddings(video_path, all_merged_detections, embedder, brightness_threshold,
                        crop_kwargs, n_sample_frames=120, total_frames=None,
                        dark_team_id=0, light_team_id=1):
    crops, brightnesses = [], []
    step  = max(1, min(4, n_sample_frames // 30))
    upper = min(n_sample_frames, total_frames or 10**9, len(all_merged_detections))
    cap_s = cv2.VideoCapture(video_path)

    for fi in range(0, upper, step):
        cap_s.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ret, sf = cap_s.read()
        if not ret:
            continue
        for det in all_merged_detections[fi]:
            if det["class"] != "player":
                continue
            c = crop_player(sf, det["xyxy"])
            v = extract_jersey_brightness(sf, det["xyxy"], **crop_kwargs)
            if c is not None and v is not None:
                crops.append(c)
                brightnesses.append(v)
    cap_s.release()

    if len(crops) < 10 or brightness_threshold is None:
        print(f"  Not enough ReID samples ({len(crops)}).")
        return None, {}

    embs = embedder.embed(crops).numpy()
    km   = KMeans(n_clusters=2, n_init=10, random_state=42).fit(embs)
    centroids = F.normalize(torch.from_numpy(km.cluster_centers_).float(), dim=1)

    brightnesses = np.array(brightnesses)
    cluster_brightness = [brightnesses[km.labels_ == k].mean() for k in range(2)]

    # ── Quality gate: do the ReID clusters actually separate teams? ──
    # Compare the brightness gap between ReID clusters to the GMM gap.
    # If the ReID clusters are both on the same side of the brightness
    # threshold (or the gap is tiny), the embeddings didn't capture
    # jersey color — reject and fall back to brightness-only.
    reid_gap = abs(cluster_brightness[1] - cluster_brightness[0])

    # Compute GMM gap from the brightness samples for comparison
    below = brightnesses[brightnesses <= brightness_threshold]
    above = brightnesses[brightnesses >  brightness_threshold]
    if len(below) > 5 and len(above) > 5:
        gmm_gap = float(above.mean() - below.mean())
    else:
        gmm_gap = reid_gap + 1  # can't validate — give ReID benefit of doubt

    # Gate: ReID cluster gap must be at least 40% of the brightness gap
    MIN_REID_GAP_RATIO = 0.40
    if reid_gap < gmm_gap * MIN_REID_GAP_RATIO:
        print(f"  ReID cluster brightness gap ({reid_gap:.1f}) is too small "
              f"vs GMM gap ({gmm_gap:.1f}) — ReID did NOT separate teams.")
        print(f"  Falling back to brightness-only classification.")
        return None, {}

    # Label clusters RELATIVE to each other (darker cluster -> dark team,
    # lighter cluster -> light team), not against the absolute per-crop
    # threshold, which is fit on noisy single-crop samples.
    order = np.argsort(cluster_brightness)   # ascending: darker cluster first
    team_map = {int(order[0]): dark_team_id, int(order[1]): light_team_id}

    print(f"  ReID fitted on {len(crops)} crops.")
    print(f"  Cluster mean brightness: {[round(b,1) for b in cluster_brightness]}")
    print(f"  Cluster brightness gap: {reid_gap:.1f} (GMM gap: {gmm_gap:.1f}, "
          f"ratio: {reid_gap/gmm_gap:.2f}) — PASS")
    print(f"  Cluster -> team map: {team_map}")
    return centroids, team_map


def classify_by_embedding(emb, centroids, team_map, margin_thresh=0.05):
    sims   = (centroids @ emb).numpy()
    order  = np.argsort(-sims)
    margin = float(sims[order[0]] - sims[order[1]])
    if margin < margin_thresh:
        return -1, margin
    return team_map.get(int(order[0]), -1), margin


# ══════════════════════════════════════════════════════════
#  Spatial — advisory only (balance veto + diagnostics)
# ══════════════════════════════════════════════════════════

class SpatialConsistency:
    def __init__(self, window=60, expected_per_team=5):
        self.window          = window
        self.expected        = expected_per_team
        self.balance_hist    = deque(maxlen=window)
        self.dispersion_hist = deque(maxlen=window)

    @staticmethod
    def _dispersion(points):
        if len(points) < 2:
            return None
        P  = np.array(points, dtype=np.float64)
        d  = np.linalg.norm(P[:, None, :] - P[None, :, :], axis=-1)
        iu = np.triu_indices(len(P), k=1)
        return float(d[iu].mean())

    def update(self, detections, player_team_ids, possessing_team_id):
        by_team = defaultdict(list)
        for xyxy, tid in zip(detections.xyxy, player_team_ids):
            if tid is None or tid < 0:
                continue
            by_team[tid].append(((xyxy[0] + xyxy[2]) / 2, xyxy[3]))

        self.balance_hist.append({t: len(p) for t, p in by_team.items()})

        if possessing_team_id is not None and possessing_team_id >= 0:
            teams = list(by_team.keys())
            if len(teams) == 2:
                dfn = [t for t in teams if t != possessing_team_id]
                if dfn:
                    d_off = self._dispersion(by_team[possessing_team_id])
                    d_def = self._dispersion(by_team[dfn[0]])
                    if d_off is not None and d_def is not None:
                        self.dispersion_hist.append(d_off > d_def)

    def report(self):
        out = {"balance_ok": None, "dispersion_ok": None, "possible_flip": False}
        if self.balance_hist:
            bad = sum(1 for c in self.balance_hist
                      if any(abs(n - self.expected) > 1 for n in c.values()))
            out["balance_ok"] = (bad / len(self.balance_hist)) < 0.25
        if len(self.dispersion_hist) >= 20:
            rate = sum(self.dispersion_hist) / len(self.dispersion_hist)
            out["dispersion_ok"] = rate > 0.5
            out["possible_flip"] = rate < 0.30
        return out

    def current_counts(self, team_assignments):
        return Counter(team_assignments.values())

    def would_worsen_balance(self, team_assignments, proposed_team_id, expected_per_team=None):
        """
        Veto a lock if the PROPOSED team already has as many (or more)
        locked players than expected, AND some other team currently has
        room (fewer than expected). Compares against a fixed absolute
        expectation rather than the other team's live count, so it still
        resists a 6th+ lock onto one team even while the other team has zero.
        """
        expected = expected_per_team if expected_per_team is not None else self.expected
        current = dict(self.current_counts(team_assignments))
        proposed_team_count = current.get(proposed_team_id, 0)

        other_counts = [v for k, v in current.items() if k != proposed_team_id]
        other_has_room = (not other_counts) or (min(other_counts) < expected)

        if proposed_team_count >= expected and other_has_room:
            return True
        return False

In [ ]:
VIDEO_PATH_REAL = "VIDEO_PATH!"
OUTPUT_VIDEO_REAL = "OUTPUT_VIDEO_PATH"

# Get video dimensions
_, W, H, _ = get_video_info(VIDEO_PATH_REAL)
frame_width = W
frame_height = H

print(f"Detected video dimensions: {frame_width}x{frame_height}")

Detected video dimensions: 2868x1320


MAIN PIPELINE
---------------------------


In [ ]:
"""CELL 14 — Main pipeline (run_analysis)
------------------------------------------------------------------------------
Only one copy of this function should exist in your notebook. If you paste
this in, delete every other `def run_analysis(...)` cell you have — Python
silently uses whichever one is defined LAST, so a leftover older copy below
this one will quietly take over again.
"""

def run_analysis(video_path="VIDEO PATH", output_path="OUTPUT PATH",
                 enable_team_colors=True, n_sample_frames=120,
                 use_reid=True, margin_thresh=0.05,
                 dark_team_id=0, light_team_id=1,
                 run_crop_diagnostic=True,
                 use_gmm_threshold=True,
                 vote_confidence_thresh=0.75,
                 use_balance_veto=True,
                 expected_per_team=5,
                 conf_player=0.20,
                 conf_ball=0.15,
                 enable_court_features=True,
                 court_update_every=8):

    fps, W, H, total_frames = get_video_info(video_path)
    print(f"Video: {total_frames} frames @ {fps:.1f} fps  |  {W}x{H}")

    cap = cv2.VideoCapture(video_path)
    out = cv2.VideoWriter(output_path,
                          cv2.VideoWriter_fourcc(*"mp4v"),
                          fps, (W, H))

    tracker = sv.ByteTrack()

    # ── Ball tracking state ──
    ball_trail     = []
    ball_track_fi  = []
    ball_track_xs  = []
    ball_track_ys  = []
    rim_bbox       = None
    shot_arc_pts   = None
    arc_color      = ARC_COLOR_LIVE
    overlay_text   = None
    overlay_color  = (255, 255, 255)
    overlay_frames = 0
    shot_log       = []
    frame_idx      = 0

    # ── Team assignment state ──
    team_assignments    = {}
    team_vote_buffer    = {}    # kept for ReID path (rarely used)
    team_confirm_count  = {}
    brightness_samples  = {}    # track_id -> [brightness values]
    gold_locked         = set() # tracks confirmed by 5v5 consensus
    track_last_seen     = {}    # track_id -> last frame_idx seen

    # ── Court state ──
    court_transformer   = None
    court_kp_store = CourtKeypointStore(
        alpha=COURT_EMA_ALPHA,
        max_age=COURT_MAX_AGE_FRAMES
    )

    # ── Constants ──
    LOCK_THRESHOLD          = 8        # ReID vote buffer size
    CONFIRM_THRESHOLD       = 10       # frames before drawing a player
    BRIGHTNESS_LOCK_SAMPLES = 30       # accumulate before locking
    GOLD_LOCK_ABSENT_FRAMES = 90       # gold lock expires after ~1.5s absence
    POSSESSION_RADIUS_PX    = 120
    MAX_TRACK               = 90       # max ball track history length

    # ── Pre-processing ──
    print("Running merged predictions...")
    all_merged_detections = predict_merged(video_path, conf_player=conf_player, conf_ball=conf_ball)
    print("Merged predictions complete.")

    TEAM_DRAW_COLORS = {
        dark_team_id:  (40, 60, 30),     # dark green, BGR
        light_team_id: (230, 230, 230),  # white, BGR
        -1:            (160, 160, 160),
    }

    # ── Crop-region diagnostic (once) ──
    if run_crop_diagnostic and enable_team_colors:
        print("Running crop-region diagnostic...")
        crop_kwargs = diagnose_crop_regions(
            video_path, all_merged_detections,
            n_sample_frames=n_sample_frames, total_frames=total_frames
        )
    else:
        crop_kwargs = {"torso_top": 0.15, "torso_bottom": 0.55, "center_frac": 0.6}

    # ── Fit brightness threshold (GMM) ──
    brightness_threshold = None
    if enable_team_colors:
        print("Fitting brightness threshold...")
        brightness_threshold = fit_brightness_threshold(
            video_path, all_merged_detections, crop_kwargs,
            n_sample_frames=n_sample_frames, total_frames=total_frames,
            use_gmm=use_gmm_threshold
        )

    # ── Fit ReID (secondary) ──
    reid_centroids, reid_team_map, embedder = None, {}, None
    if enable_team_colors and use_reid and brightness_threshold is not None:
        print("Fitting ReID team classifier...")
        embedder = ReIDEmbedder()
        reid_centroids, reid_team_map = fit_team_embeddings(
            video_path, all_merged_detections, embedder, brightness_threshold,
            crop_kwargs, n_sample_frames=n_sample_frames, total_frames=total_frames,
            dark_team_id=dark_team_id, light_team_id=light_team_id
        )

    spatial = SpatialConsistency(window=60, expected_per_team=expected_per_team)

    # ═══════════════════════════════════════════════════════════════════
    #  MAIN LOOP
    # ═══════════════════════════════════════════════════════════════════
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_merged_dets            = all_merged_detections[frame_idx]
        player_detections_xyxy       = []
        player_detections_confidence = []
        player_detections_class_id   = []
        best_ball                    = None
        all_ball_dets                = []
        current_rim_bbox             = None

        for det in frame_merged_dets:
            if det["class"] == "player":
                player_detections_xyxy.append(det["xyxy"])                                 #if you do not want player tracking comment this line out
                player_detections_confidence.append(det["conf"])                           #if you do not want player tracking comment this line out
                player_detections_class_id.append(1)                                       #if you do not want player tracking comment this line out
            elif det["class"] == "ball":
                all_ball_dets.append((det["center"][0], det["center"][1], det["conf"]))
            elif det["class"] == "rim":
                current_rim_bbox = det["xyxy"]

        # ── Court keypoints — minimap / analytics ONLY ──
        if enable_court_features and frame_idx % court_update_every == 0:
            kpts = get_court_keypoints(frame)
            court_kp_store.update(kpts, frame_idx)
            persisted = court_kp_store.get_all()
            new_transformer = build_court_transformer(persisted)
            if new_transformer is not None:
                court_transformer = new_transformer

        # ── Build supervision Detections ──
        if not player_detections_xyxy:
            xyxy_array       = np.empty((0, 4))
            confidence_array = np.empty((0,))
            class_id_array   = np.empty((0,), dtype=int)
        else:
            xyxy_array       = np.array(player_detections_xyxy)
            confidence_array = np.array(player_detections_confidence)
            class_id_array   = np.array(player_detections_class_id)

        detections = sv.Detections(
            xyxy       = xyxy_array,
            confidence = confidence_array,
            class_id   = class_id_array
        )
        detections = tracker.update_with_detections(detections)

        # ── Track visibility & gold lock expiry ──
        visible_track_ids = set()
        if len(detections) > 0:
            for _t in detections.tracker_id:
                _t = int(_t)
                track_last_seen[_t] = frame_idx
                visible_track_ids.add(_t)

        expired_gold = [t for t in gold_locked
                        if frame_idx - track_last_seen.get(t, 0) > GOLD_LOCK_ABSENT_FRAMES]
        for t in expired_gold:
            gold_locked.discard(t)
            team_assignments.pop(t, None)
            brightness_samples.pop(t, None)

        # ── Team assignment ──
        classifier_ready = (reid_centroids is not None) or (brightness_threshold is not None)

        if classifier_ready and len(detections) > 0:
            player_team_ids = [None] * len(detections)
            pending_idx, pending_crops = [], []

            for i, (xyxy, track_id) in enumerate(zip(detections.xyxy, detections.tracker_id)):
                track_id = int(track_id)

                # ── Already locked? ──
                if track_id in team_assignments:
                    player_team_ids[i] = team_assignments[track_id]
                    continue

                # ── ReID path (if available) ──
                if reid_centroids is not None:
                    c = crop_player(frame, xyxy.astype(int))
                    if c is None:
                        player_team_ids[i] = -1
                    else:
                        pending_idx.append(i)
                        pending_crops.append(c)
                        continue

                # ── Brightness path: track-level accumulation ──
                else:
                    v = extract_jersey_brightness(frame, xyxy.astype(int), **crop_kwargs)
                    if v is not None:
                        brightness_samples.setdefault(track_id, []).append(v)

                    samples = brightness_samples.get(track_id, [])

                    # Lock after enough samples — median of 30+ is very stable
                    if len(samples) >= BRIGHTNESS_LOCK_SAMPLES and track_id not in team_assignments:
                        med_v = float(np.median(samples))
                        team_id = light_team_id if med_v > brightness_threshold else dark_team_id
                        if not (use_balance_veto and
                                spatial.would_worsen_balance(team_assignments, team_id)):
                            team_assignments[track_id] = team_id

                    # Assign: locked > running median > unknown
                    if track_id in team_assignments:
                        player_team_ids[i] = team_assignments[track_id]
                    elif len(samples) >= 3:
                        med_v = float(np.median(samples))
                        player_team_ids[i] = light_team_id if med_v > brightness_threshold else dark_team_id
                    else:
                        player_team_ids[i] = -1

            # ── ReID batch (rarely active with quality gate) ──
            if pending_crops:
                embs = embedder.embed(pending_crops)
                for j, i in enumerate(pending_idx):
                    track_id = int(detections.tracker_id[i])
                    team_id, margin = classify_by_embedding(
                        embs[j], reid_centroids, reid_team_map, margin_thresh
                    )
                    conf = min(1.0, margin / (margin_thresh * 4)) if team_id >= 0 else 0.0

                    team_vote_buffer.setdefault(track_id, []).append((team_id, conf))
                    team_vote_buffer[track_id] = team_vote_buffer[track_id][-LOCK_THRESHOLD:]
                    buf = team_vote_buffer[track_id]

                    tentative = team_id
                    if len(buf) == LOCK_THRESHOLD:
                        tally = defaultdict(float)
                        for tid_vote, c in buf:
                            if tid_vote >= 0:
                                tally[tid_vote] += c
                        if tally:
                            winner = max(tally, key=tally.get)
                            total_conf = sum(tally.values())
                            tentative = winner
                            if total_conf > 0 and tally[winner] / total_conf > vote_confidence_thresh:
                                if not (use_balance_veto and
                                        spatial.would_worsen_balance(team_assignments, winner)):
                                    team_assignments[track_id] = winner

                    player_team_ids[i] = team_assignments.get(track_id, tentative)
        else:
            player_team_ids = [-1] * len(detections)

        # ── Ball candidate selection ──
        if all_ball_dets:
            if ball_trail:
                last_cx, last_cy = ball_trail[-1]
                candidates = [
                    (cx, cy, conf) for (cx, cy, conf) in all_ball_dets
                    if math.hypot(cx - last_cx, cy - last_cy) <= MAX_BALL_JUMP_PX
                ]
                if candidates:
                    best_ball = max(candidates, key=lambda x: x[2])[:2]
                else:
                    best_ball = max(all_ball_dets, key=lambda x: x[2])[:2]
                    ball_trail.clear()
                    ball_track_fi.clear()
                    ball_track_xs.clear()
                    ball_track_ys.clear()
            else:
                best_ball = max(all_ball_dets, key=lambda x: x[2])[:2]

        ball_det = best_ball

        # ── Project ball into court space (for the radar) ──
        ball_court_xy = None
        if enable_court_features and court_transformer is not None and ball_det:
            ball_court_xy = tuple(
                court_transformer.transform_points(np.array([[ball_det[0], ball_det[1]]]))[0]
            )

        # ── Rim smoothing ──
        if current_rim_bbox is not None:
            new_rim = list(map(float, current_rim_bbox))
            if rim_bbox is None:
                rim_bbox = new_rim
            else:
                rim_bbox = [
                    RIM_SMOOTH_ALPHA * new_rim[k] + (1 - RIM_SMOOTH_ALPHA) * rim_bbox[k]
                    for k in range(4)
                ]

        # ── Ball trail + track history ──
        if ball_det:
            cx, cy = ball_det
            ball_trail.append((cx, cy))
            if len(ball_trail) > TRAIL_LENGTH:
                ball_trail.pop(0)
            ball_track_fi.append(frame_idx)
            ball_track_xs.append(cx)
            ball_track_ys.append(cy)
            if len(ball_track_fi) > MAX_TRACK:
                ball_track_fi = ball_track_fi[-MAX_TRACK:]
                ball_track_xs = ball_track_xs[-MAX_TRACK:]
                ball_track_ys = ball_track_ys[-MAX_TRACK:]

        # ── Possession ──
        last_possessor_track_id = None
        last_possessor_team_id  = None

        if ball_det and len(detections) > 0:
            ball_cx, ball_cy = ball_det
            min_dist = float("inf")
            for i, xyxy in enumerate(detections.xyxy):
                px   = (xyxy[0] + xyxy[2]) / 2
                py   = xyxy[3]
                dist = math.hypot(px - ball_cx, py - ball_cy)
                if dist < min_dist and dist < POSSESSION_RADIUS_PX:
                    min_dist                = dist
                    last_possessor_track_id = int(detections.tracker_id[i])
                    last_possessor_team_id  = player_team_ids[i]

        spatial.update(detections, player_team_ids, last_possessor_team_id)

        # ── 5v5 consensus gold lock at checkpoint ──
        if frame_idx > 0 and frame_idx % 60 == 0:
            rep = spatial.report()

            dark_now  = sum(1 for t in visible_track_ids
                           if team_assignments.get(t) == dark_team_id)
            light_now = sum(1 for t in visible_track_ids
                           if team_assignments.get(t) == light_team_id)

            if dark_now == expected_per_team and light_now == expected_per_team:
                newly = 0
                for t in visible_track_ids:
                    if t in team_assignments and t not in gold_locked:
                        gold_locked.add(t)
                        newly += 1
                if newly > 0:
                    print(f"  [frame {frame_idx}] 5v5 confirmed — "
                          f"gold-locked {newly} new tracks "
                          f"({len(gold_locked)} total)")
            else:
                if rep["possible_flip"]:
                    print(f"  [frame {frame_idx}] WARN: dispersion inverted — "
                          f"team labels may be globally flipped.")
                if rep["balance_ok"] is False:
                    print(f"  [frame {frame_idx}] WARN: team counts off 5v5 "
                          f"({dark_now}v{light_now})")

        # ── Shot detection ──
        if len(ball_track_fi) >= MIN_TRACK_LEN:
            arc_height = max(ball_track_ys) - min(ball_track_ys)
            if len(ball_track_fi) >= 3:
                _, _, _, r_sq = fit_parabola(ball_track_fi, ball_track_ys, fps)
                if arc_height >= MIN_ARC_HEIGHT_PX and r_sq >= MIN_R_SQUARED:
                    arc_pts, poly = predict_arc_points(
                        ball_track_fi, ball_track_xs, ball_track_ys, fps
                    )
                    shot_arc_pts = arc_pts

                    if rim_bbox is not None and overlay_frames == 0:
                        # ── Proximity gate: only check when ball is near rim ──
                        rim_top = rim_bbox[1]
                        rim_h   = rim_bbox[3] - rim_bbox[1]
                        ball_latest_y = ball_track_ys[-1]

                        if ball_latest_y >= rim_top - rim_h * 3:
                            mock_track = [
                                (ball_track_fi[k], ball_track_xs[k], ball_track_ys[k], [0,0,0,0], 0.0)
                                for k in range(len(ball_track_fi))
                            ]
                            made, method, made_frame = is_made_shot(
                                mock_track, poly, fps,
                                list(map(int, rim_bbox)),
                                np.array([(fi - ball_track_fi[0]) / fps for fi in ball_track_fi]),
                                ball_track_xs
                            )
                            if made_frame is not None:
                                label          = "MADE" if made else "MISSED"
                                arc_color      = ARC_COLOR_MAKE if made else ARC_COLOR_MISS
                                overlay_text   = label
                                overlay_color  = arc_color
                                overlay_frames = OVERLAY_DURATION
                                shot_log.append({
                                    "frame":      made_frame,
                                    "result":     label,
                                    "ball_pos":   [ball_track_xs[-1], ball_track_ys[-1]],
                                    "rim_bbox":   list(map(int, rim_bbox)),
                                    "shooter_id": last_possessor_track_id,
                                    "team_id":    last_possessor_team_id,
                                })
                                print(f"  Shot detected @ frame {made_frame}: {label} "
                                      f"(method={method}, team {last_possessor_team_id})")
                                ball_track_fi.clear()
                                ball_track_xs.clear()
                                ball_track_ys.clear()
                                shot_arc_pts = None

        # ── Draw players + collect radar points ──
        radar_points = []
        for i, (xyxy, track_id, team_id) in enumerate(
            zip(detections.xyxy, detections.tracker_id, player_team_ids)
        ):
            track_id = int(track_id)
            x1, y1, x2, y2 = map(int, xyxy)

            prev = team_confirm_count.get(track_id, (None, 0))
            if prev[0] == team_id:
                team_confirm_count[track_id] = (team_id, prev[1] + 1)
            else:
                team_confirm_count[track_id] = (team_id, 1)

            if team_confirm_count[track_id][1] < CONFIRM_THRESHOLD:
                continue

            if enable_court_features and court_transformer is not None:
                foot_xy = court_transformer.transform_points(
                    np.array([[(x1 + x2) / 2, y2]])
                )[0]
                radar_points.append({"court_xy": tuple(foot_xy), "team": team_id, "track_id": track_id})

            color = TEAM_DRAW_COLORS.get(team_id, (160, 160, 160))
            label = "REF" if team_id == -1 else f"T{team_id} #{track_id}"
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, label, (x1, y1 - 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2, cv2.LINE_AA)

        # ── Draw rim ──
        if rim_bbox is not None:
            x1, y1, x2, y2 = map(int, rim_bbox)
            cv2.rectangle(frame, (x1, y1), (x2, y2), RIM_COLOR, 2)
            cv2.putText(frame, "RIM", (x1, y1 - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, RIM_COLOR, 2)

        # ── Draw ball trail + arc + overlay ──
        draw_fading_trail(frame, ball_trail)

        if shot_arc_pts:
            draw_arc(frame, shot_arc_pts, arc_color)

        if overlay_frames > 0:
            alpha = min(1.0, overlay_frames / 30.0)
            draw_make_miss_banner(frame, overlay_text, overlay_color, alpha)
            overlay_frames -= 1
            if overlay_frames == 0:
                shot_arc_pts = None
                arc_color    = ARC_COLOR_LIVE

        # ── Composite radar minimap ──
        if enable_court_features and court_transformer is not None:
            radar_img = draw_radar_frame(radar_points, ball_court_xy, TEAM_DRAW_COLORS)
            rh, rw = radar_img.shape[:2]
            y0, x0 = RADAR_MARGIN, W - rw - RADAR_MARGIN
            frame[y0:y0+rh, x0:x0+rw] = cv2.addWeighted(
                frame[y0:y0+rh, x0:x0+rw], 0.15, radar_img, 0.85, 0
            )

        cv2.putText(frame, f"Frame {frame_idx}", (12, H - 14),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180, 180, 180), 1)

        out.write(frame)
        frame_idx += 1

        if frame_idx % 100 == 0:
            print(f"  Processed {frame_idx}/{total_frames} frames...")

    # ═══════════════════════════════════════════════════════════════════
    #  CLEANUP + OUTPUT
    # ═══════════════════════════════════════════════════════════════════
    cap.release()
    out.release()
    print(f"\nDone. Output saved to: {output_path}")

    final_rep = spatial.report()
    print(f"Spatial consistency: {final_rep}")

    log_path = str(Path(output_path).with_suffix(".json"))
    with open(log_path, "w") as f:
        json.dump({
            "video": Path(video_path).name,
            "fps": fps,
            "shots": shot_log,
            "spatial_report": final_rep,
            "team_assignments": {str(k): int(v) for k, v in team_assignments.items()},
        }, f, indent=2)
    print(f"Shot log saved to: {log_path}")

    return shot_log

RUN FOR ANALYSIS
=====================


In [ ]:

import gc
import torch

# Free GPU memory before running inference
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    print(f"GPU memory before analysis: {torch.cuda.memory_allocated()/1e9:.2f} GB used")

shot_results = run_analysis("/content/JBpostfade.mov", "/content/JBpostfadeOVERLAY.mov")

# Clean up GPU memory after
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    print(f"GPU memory after analysis: {torch.cuda.memory_allocated()/1e9:.2f} GB used")

print(f"\n=== Summary: {len(shot_results)} shot(s) detected ===")
for i, s in enumerate(shot_results):
    print(f"  Shot {i+1}: {s['result']}  (frame {s['frame']})")